Step 1: Extract Images Without Alt Text

In [ ]:
import requests
import torch
import clip
import pandas as pd
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO
from urllib.parse import urljoin
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Base URL
BASE_URL = "https://www.magicalmelghat.in"

# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# Use a session for efficiency
session = requests.Session()

def generate_alt_text(image_url):
    """Generate alternative text using OpenAI CLIP model."""
    try:
        response = session.get(image_url, timeout=5)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")

        image_input = preprocess(image).unsqueeze(0).to(device)

        # Define generic text descriptions
        text_descriptions = [
            "A scenic landscape",
            "A person in traditional clothing",
            "An animal in the wild",
            "A historical monument",
            "A group of people at an event",
            "A food dish"
        ]
        
        text_inputs = clip.tokenize(text_descriptions).to(device)

        # Compute similarity
        with torch.no_grad():
            image_features = model.encode_image(image_input)
            text_features = model.encode_text(text_inputs)
            similarity = (image_features @ text_features.T).softmax(dim=-1)
            best_match_idx = similarity.argmax().item()

        return text_descriptions[best_match_idx]

    except (requests.exceptions.RequestException, IOError) as e:
        print(f"❌ Error fetching image: {image_url} -> {e}")
        return "Image unavailable"

# Scrape the website
try:
    response = session.get(BASE_URL, timeout=5, verify=False)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
except requests.exceptions.RequestException as e:
    print(f"❌ Failed to fetch webpage: {BASE_URL} -> {e}")
    exit()

# Extract image details
image_data = []
for img in soup.find_all("img"):
    src = img.get("src")
    alt = img.get("alt")

    if src and not alt:  # Process only missing alt text
        full_url = urljoin(BASE_URL, src)
        image_data.append({"image_url": full_url, "generated_alt": ""})

# Generate alt text for images
for item in image_data:
    item["generated_alt"] = generate_alt_text(item["image_url"])

# Save results to CSV
df = pd.DataFrame(image_data)
df.to_csv("review_alt_text_report.csv", index=False, encoding="utf-8")

# Update HTML with generated alt text
for img in soup.find_all("img"):
    src = img.get("src")
    if src:
        full_url = urljoin(BASE_URL, src)
        for item in image_data:
            if item["image_url"] == full_url:
                img["alt"] = item["generated_alt"]
                break

# Save the updated HTML page
with open("updated_page.html", "w", encoding="utf-8") as file:
    file.write(str(soup))

print("Process completed! Alt text generated and saved.")


d:\python3.12\Lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


✅ Process completed! Alt text generated and saved.
